In [2]:
!pip install cvxpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 75.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 64.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 55.1 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.2 -> 24.3.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import norm, multivariate_normal
from scipy.optimize import minimize
import os
import cvxpy as cp

################################################################################
# Kommentarer om relevans af metoder:
# - Simulation af ΔX_t: Inspireret af Ordinary Exam 2023, hvor vi simulerer logreturns
#   og andre market invariants med np.random.multivariate_normal.
# - Obligationer og ZCB priser: Inspireret af Retake Exam 2022, hvor yield curves (Nelson-Siegel-Svensson)
#   og CashFlow-objekter anvendes til at beregne obligationspriser.
# - Porteføljeoptimering (Mean-Variance, CVaR, hedge ratio): Ordinary Exam 2021 og Retake Exam 2022 viser
#   eksempler på Mean-Variance optimering, CVaR optimering og hedge ratio beregning.
# - Visualisering: Fan charts, histogrammer og sammenligning med analytiske fordelinger hentes fra
#   diverse tidligere eksamener og kursusmateriale.
################################################################################

########################################
# Hjælpefunktioner til dataload
########################################
def load_data(data_dir="data"):
    init_values_path = os.path.join(data_dir, "init_values.xlsx")
    cov_path = os.path.join(data_dir, "covariance_matrix.xlsx")
    
    init_values = pd.read_excel("init_values.xlsx", index_col=0)
    covariance_matrix = pd.read_excel("covariance_matrix.xlsx", index_col=0)
    
    return init_values, covariance_matrix

########################################
# Funktion til at simulere ΔX_t
########################################
def simulate_X(T=52, mu=None, Sigma=None, x0=None, seed=42):
    """
    Simulerer udviklingen i X_t over T uger.
    X_t indeholder log(FX), log(V_US,local), log(V_EUR), yields mm.
    Inspireret af standard simulationsteknik fra Ordinary Exam 2023.
    """
    np.random.seed(seed)
    dim = len(mu)
    X = np.zeros((T+1, dim))
    X[0,:] = x0.values.ravel()
    for t in range(1, T+1):
        delta = np.random.multivariate_normal(mu, Sigma)
        X[t,:] = X[t-1,:] + delta
    return X

########################################
# Analysefunktioner: sammenligning af distributioner
########################################
def compare_distribution(sim_data, mu, sigma, var_index=0, plot_path="results/plots/distribution_comparison.png"):
    """
    Sammenligner en simuleret distribution (histogram) med en teoretisk normalfordeling.
    Inspireret af Ordinary Exam 2021 og Retake Exam 2021, hvor simuleret data sammenlignes
    med teoretisk pdf.
    """
    plt.figure(figsize=(6,4))
    x_values = np.linspace(np.min(sim_data[:,var_index]), np.max(sim_data[:,var_index]), 200)
    pdf_vals = norm.pdf(x_values, loc=mu[var_index], scale=np.sqrt(sigma[var_index,var_index]))
    plt.hist(sim_data[:,var_index], bins=30, density=True, alpha=0.5, label="Simulated")
    plt.plot(x_values, pdf_vals, 'r-', label="Theoretical N")
    plt.xlabel("Value")
    plt.ylabel("Density")
    plt.title("Distribution Comparison")
    plt.legend()
    os.makedirs(os.path.dirname(plot_path), exist_ok=True)
    plt.savefig(plot_path)
    plt.close()

########################################
# Funktioner til obligationspriser
# Antagelser: Vi har yields i init_values, brug lineær interpolation eller givne parametre.
# For simplicity, antag at yields er givet direkte fra X_path (f.eks. EUR og USD yields).
########################################

def zcb_price(y, T):
    """
    Pris på zero-coupon bond: Z = exp(-y*T)
    y er continuously compounded yield, T er tid til maturity.
    """
    return np.exp(-y*T)

def simulate_bond_prices(X_path, idx_yEUR5, idx_yUSD5, horizon=1):
    """
    Beregn pris på en 5-års ZCB i EUR og en 5-års ZCB i USD ved tid 0 og efter 1 år (fx).
    Inspireret af Retake Exam 2022, hvor vi bruger yields fra X til at prissætte obligationspriser.
    
    idx_yEUR5 og idx_yUSD5 er indeksene for yields for 5y EUR og USD i X-vektoren.
    horizon i år (fx 1 år) -> T=5 år initial, efter 1 år er den ZCB 4 år til maturity.
    """
    # Price initial (t=0):
    yEUR5_0 = X_path[0, idx_yEUR5]
    yUSD5_0 = X_path[0, idx_yUSD5]
    ZCB_EUR_5Y_0 = zcb_price(yEUR5_0, 5.0)
    ZCB_USD_5Y_0 = zcb_price(yUSD5_0, 5.0)
    
    # Price efter 1 år (t=52 uger):
    yEUR5_1 = X_path[-1, idx_yEUR5]
    yUSD5_1 = X_path[-1, idx_yUSD5]
    ZCB_EUR_4Y_1 = zcb_price(yEUR5_1, 4.0)
    ZCB_USD_4Y_1 = zcb_price(yUSD5_1, 4.0)
    
    return ZCB_EUR_5Y_0, ZCB_USD_5Y_0, ZCB_EUR_4Y_1, ZCB_USD_4Y_1

########################################
# Porteføljeoptimering: Mean-Variance, Hedge Ratio, CVaR
########################################

def hedge_ratio(PnL_cov, h_assets):
    """
    Beregner hedge ratio: 
    h1 = -(Σ_12 h2)/Σ_11
    Inspireret af Retake Exam 2022, hvor hedge ratio beregnes fra kovariansmatrix.
    """
    Sigma11 = PnL_cov[0,0]
    Sigma12 = PnL_cov[0,1:]
    h1 = - (Sigma12 @ h_assets) / Sigma11
    return h1

def mean_variance_optimization(mu, Sigma, target_return):
    """
    Mean-Variance optimering med no-shorts constraint.
    Inspireret af Ordinary Exam 2021.
    Minimér w^T Σ w subject to w^T 1=1, w^T mu = target_return, w>=0.
    """
    n = len(mu)
    w = cp.Variable(n, nonneg=True)
    constraints = [cp.sum(w)==1, w@mu == target_return]
    obj = cp.Minimize(cp.quad_form(w, Sigma))
    prob = cp.Problem(obj, constraints)
    prob.solve()
    return w.value

def cvar_optimization(returns, alpha=0.95, target_return=None, mu=None):
    """
    CVaR optimering:
    Inspireret af Ordinary Exam 2021, hvor CVaR minimeres under constraints.
    returns: matrix af scenarier for porteføljeafkast.
    Minimér CVaR med constraints w^T mu = target_return, sum w=1, w>=0.
    """
    # Antag returns er shape (N_sims, N_assets), vi skal finde w.
    # For simplicity, antag vi vil minimere CVaR uden specificeret target_return her.
    N, d = returns.shape
    w = cp.Variable(d, nonneg=True)
    alpha_var = cp.Variable()
    z = cp.Variable(N, nonneg=True) # helper vars

    # Mean constraint
    constraints = [cp.sum(w)==1]
    if target_return is not None and mu is not None:
        constraints.append(w@mu >= target_return)

    # CVaR definition:
    # Min alpha + (1/(N*(1-alpha))) sum z_i
    # subject to z_i >= -(R_i w + alpha), z_i >=0
    port_returns = returns @ w
    for i in range(N):
        constraints.append(z[i] >= - (port_returns[i] + alpha_var))

    obj = cp.Minimize(alpha_var + (1.0/(N*(1-alpha))) * cp.sum(z))
    prob = cp.Problem(obj, constraints)
    prob.solve()
    return w.value

########################################
# Visualisering (Fan charts, histograms)
########################################

def fan_chart(time_points, percentiles, ax=None, color="navy", labels=None):
    """
    Fan chart plot som anvendt i flere tidligere opgaver.
    percentiles antages sorteret. f.eks. [2.5%,5%,10%,50%,90%,95%,97.5%]
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(10,6))
    mid = len(percentiles)//2
    for i in range(mid):
        ax.fill_between(time_points, percentiles[i], percentiles[-i-1], alpha=0.2)

    ax.plot(time_points, percentiles[mid], color="black")
    if labels is not None:
        ax.legend(labels)
    return ax

########################################
# Hovedprogram - svarer på spørgsmålene i opgaven
########################################

def main():
    # 1. Load data
    init_values, covariance_matrix = load_data()
    
    # Ifølge opgave: mu = [0, 0.07*Δt, 0.06*Δt, 0,0,...]
    # Δt=1/52
    dt = 1/52
    dim = covariance_matrix.shape[0]
    mu = np.zeros(dim)
    # Antag variabler: 0: log(FX), 1: log(V_US), 2: log(V_EUR), derefter yields ...
    # fra opgavebeskrivelsen: mu[1] = 0.07*dt, mu[2]=0.06*dt.
    mu[1] = 0.07*dt
    mu[2] = 0.06*dt

    Sigma = covariance_matrix.values
    x0 = init_values.iloc[:,0] # initiale værdier
    
    # 2. Simuler ΔX_t
    T=52
    X = simulate_X(T=T, mu=mu, Sigma=Sigma, x0=x0)

    # Visualiser log(FX_t)
    log_fx = X[:,0]
    plt.figure(figsize=(10,4))
    plt.plot(log_fx, label="log(FX_t)")
    plt.title("Evolution of log(FX_t)")
    plt.xlabel("Week")
    plt.ylabel("log(FX)")
    os.makedirs("results/plots", exist_ok=True)
    plt.savefig("results/plots/log_fx_evolution.png")
    plt.close()

    # 3. Distribution V_US,local: sammenlign simuleret med analytisk
    # Her bruges compare_distribution på fx V_US (index=1)
    compare_distribution(X, mu, Sigma, var_index=1, plot_path="results/plots/VUS_distribution.png")

    # 4. Beregning af obligationspriser:
    # Find index for yEUR5 og yUSD5 i X. Antag ordning i X: efter equities ligger yields.
    # Ifølge opgavebeskrivelsen: ΔX = [ΔlogFX, ΔlogVUS, ΔlogVEUR, ΔyEUR_1M,ΔyEUR_1Y, ΔyEUR_3Y, ΔyEUR_5Y, ... ΔyUSD_5Y ...]
    # Lad os antage: yEUR5 på index 6, yUSD5 på index 12 (tilpas efter din kovariansmatrix)
    # Disse skal justeres i en rigtig løsning ift. opgavens dimensioner. Her gøres en antagelse:
    idx_yEUR5 = 6
    idx_yUSD5 = 12
    ZCB_EUR_5Y_0, ZCB_USD_5Y_0, ZCB_EUR_4Y_1, ZCB_USD_4Y_1 = simulate_bond_prices(X, idx_yEUR5, idx_yUSD5)

    # 5. Joint distribution af P = (FX_1, V_US,local, V_EUR, ZUSD_4Y, ZEUR_4Y)
    # Her antages at vi har simuleret X, så vi kan udtrække FX_1 = exp(X_1,0?), V_US_1 = exp(X_1,1?), V_EUR_1 = ...
    # Efter 1 år (t=52): 
    FX_1 = np.exp(X[-1,0])
    VUS_1 = np.exp(X[-1,1])
    VEUR_1 = np.exp(X[-1,2])
    # Z_USD_4Y_1 og Z_EUR_4Y_1 har vi fra før: ZCB_EUR_4Y_1, ZCB_USD_4Y_1

    # 6. Distribution i EUR: P^{EUR} = (1/FX, V_US, VEUR, Z_USD_4Y, Z_EUR_4Y)
    PEUR = np.array([1/FX_1, VUS_1, VEUR_1, ZCB_USD_4Y_1, ZCB_EUR_4Y_1])

    # 7. Forward pris:
    # F_1 = FX_0 * exp((yUSD_1 - yEUR_1)*1)
    # Antag index for yUSD1=10, yEUR1=4 (fiktive index, justér efter din dimension)
    yUSD1_0 = X[0,10]
    yEUR1_0 = X[0,4]
    FX_0 = np.exp(X[0,0])
    F1_theoretical = FX_0 * np.exp((yUSD1_0 - yEUR1_0)*1)
    # Sammenligning med analytisk distribution for V_US kan laves ved at plotte teoretisk pdf mod simuleret.

    # 8. PnL vector og hedge ratio:
    # PnL_1 = h^T PnL_1, PnL_1 defineret i opgaven.
    # For at beregne PnL covariance, simuler flere stier (her antages 1 sti, men ideelt multiple simulationer).
    # Her viser vi blot formel for hedge ratio.

    # Eksempel: beregn kovarians for PnL baseret på lineær approks. (Antag en række simulationer)
    # På grund af tidsbegrænsning: vi antager vi har PnL_samples (N_sims x dimension)
    # Her demonstreres kun formel for hedge ratio, reelt skal man simulere mange paths.
    # h2 = [allokering i US eq, EUR eq, USD bond, EUR bond], 
    # og PnL_cov fra simulation:
    # h1 = hedge_ratio(PnL_cov, h2_assets)

    # 9. Plot std vs expected PnL for hedge ratios og find optimal hedge ratio:
    # Ville kræve en loop over hedge_ratios og beregning af PnL distribution. Inspireret af Retake Exam 2022.

    # 10. CVaR hedge ratio:
    # Som vist i cvar_optimization funktionen.

    # 11. Simulation study for estimation uncertainty:
    # Inspireret af Retake Exam 2022 og Ordinary Exam 2023, 
    # generer samples af kovarianser og se på spredningen i hedge ratio. (Ej fuldt implementeret her)

    # 12. Porteføljeoptimeringsstrategier:
    # Mean-Variance, med og uden hedge,
    # Lignende til mean_variance_optimization og cvar_optimization ovenfor.

    # Gem resultater (fx i CSV) til brug i LaTeX og rapport:
    os.makedirs("results/tables", exist_ok=True)
    pd.DataFrame({"ZCB_EUR_5Y_0":[ZCB_EUR_5Y_0], "ZCB_USD_5Y_0":[ZCB_USD_5Y_0],
                  "ZCB_EUR_4Y_1":[ZCB_EUR_4Y_1], "ZCB_USD_4Y_1":[ZCB_USD_4Y_1]}).to_csv("results/tables/zcb_prices.csv")

    # Visualiseringer er allerede gemt ovenfor.

    print("Kørsel afsluttet. Resultater gemt i 'results/'.")

if __name__=="__main__":
    main()


Kørsel afsluttet. Resultater gemt i 'results/'.
